# 05 · SAE quality and cross-entropy evaluation

Reconstruction quality on the held-out activation test split (MSE, explained variance, L0, dead latents) and next-token cross-entropy when the SAE is spliced into layer 12: base model, full SAE reconstruction, the top-4 intervention and mean ablation.

For the K comparison, set `K` and `SAE_CKPT_PATH` to another run in `sae_runs/`; the intervention variant is skipped automatically when no selected-features file exists.

In [ ]:
# !pip install -q -U transformers accelerate pandas tqdm huggingface_hub safetensors

from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive')
HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()

from pathlib import Path
from contextlib import contextmanager
import json
import random

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'
TEXT_CORPUS_DIR = Path('/content/drive/MyDrive/Diploma/corpora2')
ACTIVATION_DATA_DIR = Path('/content/drive/MyDrive/Diploma/activation_datasets2')
SAE_OUT_DIR = Path('/content/drive/MyDrive/Diploma/sae_runs/k56')
SAE_CKPT_PATH = SAE_OUT_DIR / 'checkpoints' / 'sae_epoch_025.pt'  # K comparison: sae_runs/k{32,48,64}/...
EVAL_OUT_DIR = Path('/content/drive/MyDrive/Diploma/sae_eval_k56_refusal_features')
FEATURE_OUT_DIR = EVAL_OUT_DIR / 'refusal_features_strict_retain005'
SELECTED_FEATURES_JSON_PATH = FEATURE_OUT_DIR / 'selected_refusal_features_strict.json'
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

D_IN = 4096
D_SAE = 16384
K = 56
HOOK_LAYER = 12
TARGET_FEATURE_TOP_N = 4
CLAMP_VALUE = 0.0
SAE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

SAE_QUALITY_SPLITS = ['test']
SAE_EVAL_BATCH_TOKENS = 8192
CE_DATASETS = {'retain': TEXT_CORPUS_DIR / 'retain.jsonl', 'sae_test': TEXT_CORPUS_DIR / 'sae_test.jsonl'}
CE_VARIANTS = ['base', 'sae_recon', 'sae_intervention', 'mean_ablation']
CE_BATCH_SIZE = 4
CE_MAX_LENGTH = 256
CE_MAX_EXAMPLES = 128  # thesis results use the first 128 records of each file; None = full file

In [ ]:
def read_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_jsonl(path: Path) -> list[dict]:
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def batch_iter(items: list, batch_size: int):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def get_hidden_from_output(output):
    return output[0] if isinstance(output, tuple) else output


def replace_hidden_in_output(output, new_hidden):
    if isinstance(output, tuple):
        output = list(output)
        output[0] = new_hidden
        return tuple(output)
    return new_hidden


class TopKSAE(nn.Module):
    def __init__(self, d_in: int, d_sae: int, k: int, dtype: torch.dtype):
        super().__init__()
        self.d_in = d_in
        self.d_sae = d_sae
        self.k = k
        self.W_enc = nn.Parameter(torch.empty(d_in, d_sae, dtype=dtype))
        self.W_dec = nn.Parameter(torch.empty(d_sae, d_in, dtype=dtype))
        self.b_enc = nn.Parameter(torch.zeros(d_sae, dtype=dtype))
        self.b_dec = nn.Parameter(torch.zeros(d_in, dtype=dtype))

    def encode_topk(self, x: torch.Tensor):
        pre_acts = (x - self.b_dec) @ self.W_enc + self.b_enc
        topk_vals, topk_idx = pre_acts.topk(self.k, dim=-1)
        return F.relu(topk_vals), topk_idx

    def decode_topk(self, topk_vals: torch.Tensor, topk_idx: torch.Tensor):
        return (topk_vals.unsqueeze(-1) * self.W_dec[topk_idx]).sum(dim=1) + self.b_dec

    def encode_dense(self, x: torch.Tensor):
        topk_vals, topk_idx = self.encode_topk(x)
        feat = torch.zeros(x.shape[0], self.d_sae, device=x.device, dtype=x.dtype)
        feat.scatter_(-1, topk_idx, topk_vals)
        return feat

    def forward(self, x: torch.Tensor):
        topk_vals, topk_idx = self.encode_topk(x)
        x_recon = self.decode_topk(topk_vals, topk_idx)
        return {'x_recon': x_recon, 'topk_vals': topk_vals, 'topk_idx': topk_idx}

In [ ]:
sae = TopKSAE(D_IN, D_SAE, K, SAE_DTYPE).to(DEVICE)
ckpt = torch.load(SAE_CKPT_PATH, map_location=DEVICE, weights_only=True)
sae.load_state_dict(ckpt['model_state_dict'])
sae.eval()

if SELECTED_FEATURES_JSON_PATH.exists() and K == 56:
    selected_payload = read_json(SELECTED_FEATURES_JSON_PATH)
    TARGET_FEATURES = [int(x) for x in selected_payload['selected_features'][:TARGET_FEATURE_TOP_N]]
else:
    TARGET_FEATURES = []
    CE_VARIANTS = [v for v in CE_VARIANTS if v != 'sae_intervention']
print('TARGET_FEATURES:', TARGET_FEATURES)

In [ ]:
def list_activation_shards(split_name: str) -> list[Path]:
    return sorted((ACTIVATION_DATA_DIR / split_name).glob(f'{split_name}_acts_*.pt'))


def iter_activation_batches(split_name: str, batch_tokens: int):
    for shard_path in list_activation_shards(split_name):
        acts = torch.load(shard_path, map_location='cpu', weights_only=True)['activations']
        for start in range(0, acts.shape[0], batch_tokens):
            yield acts[start:start + batch_tokens]


@torch.inference_mode()
def evaluate_sae_quality(split_name: str) -> dict:
    n_tokens = 0
    sse = 0.0
    l0_sum = 0.0
    sum_x = torch.zeros(D_IN, dtype=torch.float64)
    sum_x2 = torch.zeros(D_IN, dtype=torch.float64)
    feature_counts = torch.zeros(D_SAE, dtype=torch.long)

    for x_cpu in tqdm(iter_activation_batches(split_name, SAE_EVAL_BATCH_TOKENS), desc=f'SAE quality: {split_name}'):
        x = x_cpu.to(DEVICE, dtype=SAE_DTYPE)
        out = sae(x)
        err = x.float() - out['x_recon'].float()
        n = x.shape[0]
        n_tokens += n
        sse += float((err * err).sum().item())
        l0_sum += float((out['topk_vals'] > 0).float().sum(dim=-1).sum().item())
        x64 = x_cpu.double()
        sum_x += x64.sum(dim=0)
        sum_x2 += (x64 * x64).sum(dim=0)
        active_idx = out['topk_idx'][out['topk_vals'] > 0].detach().cpu().long()
        if active_idx.numel():
            feature_counts.scatter_add_(0, active_idx, torch.ones_like(active_idx, dtype=torch.long))

    mean_x = sum_x / max(n_tokens, 1)
    total_variance = float((sum_x2 - n_tokens * mean_x * mean_x).sum().item())
    dead_features = int((feature_counts == 0).sum().item())
    return {
        'split': split_name,
        'n_tokens': int(n_tokens),
        'mse': sse / max(n_tokens * D_IN, 1),
        'explained_variance': 1.0 - sse / max(total_variance, 1e-12),
        'mean_l0': l0_sum / max(n_tokens, 1),
        'active_features': D_SAE - dead_features,
        'dead_features': dead_features,
        'dead_fraction': dead_features / D_SAE,
    }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
    device_map='auto',
    low_cpu_mem_usage=True,
    attn_implementation='sdpa',
)
model.eval()
INPUT_DEVICE = next(model.model.embed_tokens.parameters()).device

In [ ]:
def render_record(record: dict) -> str:
    if 'messages' in record:
        return tokenizer.apply_chat_template(record['messages'], tokenize=False, add_generation_prompt=True)
    return str(record.get('text', record.get('prompt', record.get('instruction', ''))))


def tokenize_records(records: list[dict]):
    batch = tokenizer(
        [render_record(r) for r in records],
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=CE_MAX_LENGTH,
        add_special_tokens=False,
    )
    batch = {k: v.to(INPUT_DEVICE) for k, v in batch.items()}
    labels = batch['input_ids'].clone()
    labels[batch['attention_mask'] == 0] = -100
    batch['labels'] = labels
    return batch


@torch.inference_mode()
def apply_delta_intervention_to_flat(x_flat: torch.Tensor) -> torch.Tensor:
    feat = sae.encode_dense(x_flat)
    delta = torch.zeros_like(x_flat)
    for j in TARGET_FEATURES:
        active = feat[:, j] > 0
        if active.any():
            delta[active] += (CLAMP_VALUE - feat[active, j]).unsqueeze(-1) * sae.W_dec[j].unsqueeze(0)
    return x_flat + delta


def patch_hidden(hidden: torch.Tensor, variant: str) -> torch.Tensor:
    if variant == 'base':
        return hidden
    b, s, d = hidden.shape
    x_flat = hidden.reshape(-1, d).to(DEVICE, dtype=SAE_DTYPE)
    if variant == 'sae_recon':
        patched = sae(x_flat)['x_recon']
    elif variant == 'sae_intervention':
        patched = apply_delta_intervention_to_flat(x_flat)
    elif variant == 'mean_ablation':
        patched = sae.b_dec.unsqueeze(0).expand_as(x_flat)
    else:
        raise ValueError(variant)
    return patched.reshape(b, s, d).to(hidden.device, dtype=hidden.dtype)


@contextmanager
def patch_context(variant: str):
    if variant == 'base':
        yield
        return
    def hook(module, inputs, output):
        hidden = get_hidden_from_output(output)
        return replace_hidden_in_output(output, patch_hidden(hidden, variant))
    handle = model.model.layers[HOOK_LAYER].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


@torch.inference_mode()
def ce_loss(batch: dict, variant: str) -> float:
    with patch_context(variant):
        out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=batch['labels'], use_cache=False)
    return float(out.loss.item())


def ce_recovered(base: float, patched: float, mean_ablation: float) -> float | None:
    denom = mean_ablation - base
    if abs(denom) < 1e-12:
        return None
    return 100.0 * (mean_ablation - patched) / denom


@torch.inference_mode()
def evaluate_ce_dataset(name: str, path: Path) -> dict:
    records = read_jsonl(path)
    if CE_MAX_EXAMPLES is not None:
        records = records[:CE_MAX_EXAMPLES]
    sums = {v: 0.0 for v in CE_VARIANTS}
    n_total = 0
    for batch_records in tqdm(list(batch_iter(records, CE_BATCH_SIZE)), desc=f'CE: {name}'):
        batch = tokenize_records(batch_records)
        n = len(batch_records)
        n_total += n
        for variant in CE_VARIANTS:
            sums[variant] += ce_loss(batch, variant) * n
    avg = {v: sums[v] / max(n_total, 1) for v in CE_VARIANTS}
    return {
        'dataset': name,
        'n_examples': n_total,
        'ce_base': avg['base'],
        'ce_sae_recon': avg['sae_recon'],
        'ce_sae_intervention': avg['sae_intervention'],
        'ce_mean_ablation': avg['mean_ablation'],
        'delta_ce_recon': avg['sae_recon'] - avg['base'],
        'delta_ce_intervention': avg['sae_intervention'] - avg['base'],
        'ce_recovered_recon_pct': ce_recovered(avg['base'], avg['sae_recon'], avg['mean_ablation']),
        'ce_recovered_intervention_pct': ce_recovered(avg['base'], avg['sae_intervention'], avg['mean_ablation']),
    }

In [ ]:
quality_df = pd.DataFrame([evaluate_sae_quality(split) for split in SAE_QUALITY_SPLITS])
quality_df.to_csv(EVAL_OUT_DIR / f'sae_quality_metrics_k{K}.csv', index=False)

ce_df = pd.DataFrame([evaluate_ce_dataset(name, path) for name, path in CE_DATASETS.items() if path.exists()])
ce_df.to_csv(EVAL_OUT_DIR / f'ce_metrics_k{K}.csv', index=False)

display(quality_df)
display(ce_df)